In [2]:
import pandas as pd
import numpy as np


df = pd.read_csv('dataset/synthetic_logs_train.csv')

df.head(3)

,timestamp,source,log_message,target_label
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert


In [3]:
df.source.unique()

array(['ModernCRM', 'AnalyticsEngine', 'ModernHR', 'BillingSystem',
       'ThirdPartyAPI', 'LegacyCRM'], dtype=object)

In [4]:
df.target_label.unique()

array(['HTTP Status', 'Critical Error', 'Security Alert', 'Error',
       'System Notification', 'Resource Usage', 'User Action',
       'Workflow Error', 'Deprecation Warning'], dtype=object)

### Clustering

In [5]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN


## Load pre-trained Sentence Transformer 
model = SentenceTransformer('all-MiniLM-L6-v2')

#Generate embeddings for the log messages
embeddings = model.encode(df['log_message'].tolist())
embeddings[:2]

/home/cyborg/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1117.23it/s]


array([[-1.02939598e-01,  3.35459672e-02, -2.20261216e-02,
         1.55101111e-03, -9.86925978e-03, -1.78956240e-01,
        -6.34409934e-02, -6.01761788e-02,  2.81109102e-02,
         5.99620752e-02, -1.72618423e-02,  1.43358926e-03,
        -1.49560034e-01,  3.15287360e-03, -5.66031113e-02,
         2.71685515e-02, -1.49890641e-02, -3.54037993e-02,
        -3.62936519e-02, -1.45410383e-02, -5.61495451e-03,
         8.75539407e-02,  4.55120020e-02,  2.50964053e-02,
         1.00187557e-02,  1.24267498e-02, -1.39923573e-01,
         7.68696293e-02,  3.14095095e-02, -4.15246794e-03,
         4.36902344e-02,  1.71250012e-02, -8.00950974e-02,
         5.74006066e-02,  1.89092178e-02,  8.55262130e-02,
         3.96398790e-02, -1.34371787e-01, -1.44360214e-03,
         3.06703593e-03,  1.76854044e-01,  4.44889534e-03,
        -1.69274993e-02,  2.24266425e-02, -4.35049236e-02,
         6.09034114e-03, -9.98167321e-03, -6.23973198e-02,
         1.07372757e-02, -6.04894431e-03, -7.14660957e-0

In [11]:
## Perform DBSCAN clustering
dbscan = DBSCAN(eps=0.2, min_samples=1, metric='cosine')
clusters = dbscan.fit_predict(embeddings)

## ADD cluster labels to the DataFrame
df['cluster'] = clusters

In [12]:
df[df['cluster']==1].head(10)

,timestamp,source,log_message,target_label,cluster
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,1
10,8/9/2025 18:58,ModernCRM,Email server encountered a sending fault,Error,1
217,1/22/2025 5:45,BillingSystem,Mail service encountered a delivery glitch,Error,1
248,5/2/2025 23:04,ModernHR,Service disruption caused by email sending error,Critical Error,1
265,3/30/2025 23:53,ModernCRM,Email system had a problem sending emails,Error,1
361,11/19/2025 23:06,BillingSystem,Email service experienced a sending issue,Error,1
450,10/27/2025 5:59,ThirdPartyAPI,Email delivery system encountered an error,Error,1
477,12/2/2025 10:30,AnalyticsEngine,Email transmission error caused service impact,Critical Error,1
570,11/7/2025 18:08,ThirdPartyAPI,Email service impacted by sending failure,Critical Error,1
678,4/28/2025 15:13,AnalyticsEngine,Email delivery problem affected system,Critical Error,1


In [14]:
cluster_counts = df['cluster'].value_counts()
print('number of clusters:', len(cluster_counts))

#   Get indices of large clusters
large_clusters = cluster_counts[cluster_counts > 10].index

for cluster in large_clusters:
    print(f"Cluster {cluster}:")
    print(df[df['cluster'] == cluster]['log_message'].head(5).to_string(index=False))
    print()

number of clusters: 136
Cluster 0:
nova.osapi_compute.wsgi.server [req-b9718cd8-f6...
nova.osapi_compute.wsgi.server [req-4895c258-b2...
nova.osapi_compute.wsgi.server [req-ee8bc8ba-92...
nova.osapi_compute.wsgi.server [req-f0bffbc3-5a...
nova.osapi_compute.wsgi.server [req-2bf7cfee-a2...

Cluster 5:
nova.compute.claims [req-a07ac654-8e81-416d-bfb...
nova.compute.claims [req-d6986b54-3735-4a42-907...
nova.compute.claims [req-72b4858f-049e-49e1-b31...
nova.compute.claims [req-5c8f52bd-8e3c-41f0-95a...
nova.compute.claims [req-d38f479d-9bb9-4276-968...

Cluster 11:
User User685 logged out.
 User User395 logged in.
 User User225 logged in.
User User494 logged out.
 User User900 logged in.

Cluster 13:
Backup started at 2025-05-14 07:06:55.
Backup started at 2025-02-15 20:00:19.
  Backup ended at 2025-08-08 13:06:23.
Backup started at 2025-11-14 08:27:43.
Backup started at 2025-12-09 10:19:11.

Cluster 7:
Multiple bad login attempts detected on user 85...
Multiple login failures occurred o

### Classification with Regex

In [15]:
import re
def classify_with_regex(log_message):
    regex_patterns = {
        r"User User\d+ logged (in|out).": "User Action",
        r"Backup (started|ended) at .*": "System Notification",
        r"Backup completed successfully.": "System Notification",
        r"System updated to version .*": "System Notification",
        r"File .* uploaded successfully by user .*": "System Notification",
        r"Disk cleanup completed successfully.": "System Notification",
        r"System reboot initiated by user .*": "System Notification",
        r"Account with ID .* created by .*": "User Action"
    }
    for pattern, label in regex_patterns.items():
        if re.search(pattern, log_message):
            return label
    return None

In [16]:
## test
classify_with_regex("User User123 logged in.")

'User Action'

In [18]:
classify_with_regex("Backup started at 2023-08-01 10:00:00.")

'System Notification'

In [20]:
## no result because it is not in the pattern
classify_with_regex("hello")

In [ ]:
## applying regex classification
df['regex_label'] = df['log_message'].apply(lambda x: classify_with_regex(x))
df[df['regex_label'].notnull()].head(10)        # view classified logs

,timestamp,source,log_message,target_label,cluster,regex_label
7,10/11/2025 8:44,ModernHR,File data_6169.csv uploaded successfully by us...,System Notification,4,System Notification
14,1/4/2025 1:43,ThirdPartyAPI,File data_3847.csv uploaded successfully by us...,System Notification,4,System Notification
15,5/1/2025 9:41,ModernCRM,Backup completed successfully.,System Notification,8,System Notification
18,2/22/2025 17:49,ModernCRM,Account with ID 5351 created by User634.,User Action,9,User Action
27,9/24/2025 19:57,ThirdPartyAPI,User User685 logged out.,User Action,11,User Action
30,4/26/2025 7:54,AnalyticsEngine,Backup started at 2025-05-14 07:06:55.,System Notification,13,System Notification
36,11/19/2025 13:14,BillingSystem,System reboot initiated by user User243.,System Notification,16,System Notification
44,6/20/2025 0:53,BillingSystem,Backup started at 2025-02-15 20:00:19.,System Notification,13,System Notification
48,7/1/2025 9:43,ModernHR,File data_7366.csv uploaded successfully by us...,System Notification,4,System Notification
50,1/16/2025 21:45,ThirdPartyAPI,System updated to version 3.9.1.,System Notification,21,System Notification


In [31]:
## total shape of the dataset
print('The shape of the dataset is:', df.shape[0])
print('Number of logs classified by regex:', df['regex_label'].notnull().sum())

### The remaining rows are not classified by regex
df_non_regex = df[df['regex_label'].isnull()].copy()
print('Number of logs NOT classified by regex:', df_non_regex.shape[0])


The shape of the dataset is: 2410
Number of logs classified by regex: 500
Number of logs NOT classified by regex: 1910


### Classification Stage 2: Classification using Embeddings

In [ ]:
df_non_regex.head(3)

,timestamp,source,log_message,target_label,cluster,regex_label
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,0,None
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,1,None
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,2,None


In [ ]:
## finding the target label that has the least cases in the dataset
print(df_non_regex['target_label'].value_counts(ascending=True))

target_label
Deprecation Warning       3
Workflow Error            4
Critical Error          161
Error                   177
Resource Usage          177
Security Alert          371
HTTP Status            1017
Name: count, dtype: int64


In [ ]:
## extractting target with values less than 5
print(df_non_regex['target_label'].value_counts()[df_non_regex['target_label'].value_counts() <= 5].index.tolist())

['Workflow Error', 'Deprecation Warning']


In [41]:
# finding the target label that has the small cases
df_non_regex[df_non_regex['target_label'].isin(['Workflow Error', 'Deprecation Warning'])].head()

,timestamp,source,log_message,target_label,cluster,regex_label
60,2025-10-06 16:55:23,LegacyCRM,Lead conversion failed for prospect ID 7842 du...,Workflow Error,24,None
255,2025-05-03 16:55:35,LegacyCRM,API endpoint 'getCustomerDetails' is deprecate...,Deprecation Warning,48,None
377,2025-06-24 12:16:29,LegacyCRM,Customer follow-up process for lead ID 5621 fa...,Workflow Error,62,None
1325,2025-04-17 07:33:44,LegacyCRM,Escalation rule execution failed for ticket ID...,Workflow Error,105,None
1734,2025-04-30 07:47:30,LegacyCRM,The 'ExportToCSV' feature is outdated. Please ...,Deprecation Warning,118,None


In [43]:
df_non_legacy = df_non_regex[df_non_regex['source'] != 'LegacyCRM']
## find the unique vlaues
df_non_legacy.source.unique()   

array(['ModernCRM', 'AnalyticsEngine', 'ModernHR', 'BillingSystem',
       'ThirdPartyAPI'], dtype=object)

In [47]:
## generate embeddings for log messages

model = SentenceTransformer('all-MiniLM-L6-v2')  # Lightweight embedding model
embeddings_filtered = model.encode(df_non_legacy['log_message'].tolist())

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1584.35it/s]


In [48]:
len(embeddings_filtered)

1903

In [50]:
X = embeddings_filtered
y = df_non_legacy['target_label'].values

In [51]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

report = classification_report(y_test, y_pred)
print(report)

                precision    recall  f1-score   support

Critical Error       0.91      1.00      0.95        48
         Error       0.98      0.89      0.93        47
   HTTP Status       1.00      1.00      1.00       304
Resource Usage       1.00      1.00      1.00        49
Security Alert       1.00      0.99      1.00       123

      accuracy                           0.99       571
     macro avg       0.98      0.98      0.98       571
  weighted avg       0.99      0.99      0.99       571



In [56]:
### Export the model 

import joblib

joblib.dump(clf, '../models/log_classifier.joblib')

['../models/log_classifier.joblib']